# 02 – Data Preprocessing

## Purpose

This notebook prepares the NASA Exoplanet Archive for supervised machine learning by constructing a clean, reproducible, and scientifically meaningful dataset.

Each preprocessing step is designed to preserve scientific validity while minimizing bias and preventing data leakage.

## Objectives

This notebook will:

1. Select scientifically meaningful variables.
2. Remove duplicate planetary observations.
3. Construct the binary target variable.
4. Separate predictor variables from the target.
5. Split the dataset into training and testing sets.
6. Learn preprocessing statistics using only the training data.
7. Export the processed datasets for machine learning.

In [28]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer

In [29]:
df = pd.read_csv("../data/raw/exoplanets_raw.csv", comment="#")

print(f"Dataset loaded with {len(df):,} observations.")

Dataset loaded with 39,954 observations.


## Scientific Feature Selection

Machine learning models should use variables that are scientifically meaningful rather than simply including every available measurement.

This project investigates whether measurable stellar properties can predict whether an orbiting planet is a gas giant.

The selected stellar features describe the physical characteristics of the host star and are available for a large proportion of confirmed exoplanets.

Planetary radius is retained only to construct the target variable and is removed before model training to prevent target leakage.

In [30]:
selected_columns = [
    "pl_name",
    "pl_rade",
    "st_teff",
    "st_rad",
    "st_met",
    "st_age"
]

df = df[selected_columns].copy()

print(df.head())

    pl_name  pl_rade  st_teff  st_rad  st_met  st_age
0  11 Com b      NaN   4874.0   13.76   -0.26     NaN
1  11 Com b      NaN   4742.0   19.00   -0.35     NaN
2  11 Com b      NaN      NaN     NaN     NaN     NaN
3  11 UMi b      NaN   4213.0   29.79   -0.02     NaN
4  11 UMi b      NaN      NaN     NaN     NaN     NaN


## Remove Duplicate Planetary Observations

The NASA Exoplanet Archive may contain multiple measurements for the same confirmed exoplanet because planetary parameters are refined over time and reported by different scientific studies.

Since the objective of this project is to classify planets rather than individual observations, each confirmed planet should appear only once in the machine learning dataset.

In [31]:
duplicate_count = df["pl_name"].duplicated().sum()

print(f"Duplicate planet records: {duplicate_count:,}")

df = df.drop_duplicates(subset="pl_name", keep="first")

print(f"Unique planets remaining: {len(df):,}")

Duplicate planet records: 33,638
Unique planets remaining: 6,316


In [32]:
df = df.drop(columns=["st_age"])

print(df.columns.tolist())

['pl_name', 'pl_rade', 'st_teff', 'st_rad', 'st_met']


## Defining the Target Variable

The objective of this project is to predict whether a confirmed exoplanet is a gas giant using only the properties of its host star.

A binary target variable is constructed using planetary radius. Planets with radii greater than **6 Earth radii** are classified as gas giants, while smaller planets are classified as non-gas giants.

Planetary radius is used only to create the target variable and is removed before model training to prevent target leakage.

In [33]:
df = df.dropna(subset=["pl_rade"])

df["is_gas_giant"] = (df["pl_rade"] > 6).astype(int)

print(df["is_gas_giant"].value_counts())

is_gas_giant
0    3043
1     779
Name: count, dtype: int64


In [34]:
X = df.drop(columns=["pl_name", "pl_rade", "is_gas_giant"])

y = df["is_gas_giant"]

print("Predictor variables:")
print(X.columns.tolist())

Predictor variables:
['st_teff', 'st_rad', 'st_met']


## Preventing Data Leakage

The dataset is divided into training and testing sets **before** missing values are imputed.

This ensures that preprocessing statistics are learned exclusively from the training data, preventing information from the testing set from influencing model development.

In [35]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

print(f"Training observations: {len(X_train):,}")
print(f"Testing observations: {len(X_test):,}")

Training observations: 3,057
Testing observations: 765


In [36]:
imputer = SimpleImputer(strategy="median")

X_train = pd.DataFrame(
    imputer.fit_transform(X_train),
    columns=X_train.columns,
    index=X_train.index
)

X_test = pd.DataFrame(
    imputer.transform(X_test),
    columns=X_test.columns,
    index=X_test.index
)

In [37]:
print("Missing values in training set:")
print(X_train.isnull().sum())

print("\nMissing values in testing set:")
print(X_test.isnull().sum())

Missing values in training set:
st_teff    0
st_rad     0
st_met     0
dtype: int64

Missing values in testing set:
st_teff    0
st_rad     0
st_met     0
dtype: int64


In [38]:
print("Training class distribution")
print(y_train.value_counts(normalize=True).round(3))

print("\nTesting class distribution")
print(y_test.value_counts(normalize=True).round(3))

Training class distribution
is_gas_giant
0    0.796
1    0.204
Name: proportion, dtype: float64

Testing class distribution
is_gas_giant
0    0.796
1    0.204
Name: proportion, dtype: float64


In [39]:
X_train.to_csv("../data/processed/X_train.csv", index=False)
X_test.to_csv("../data/processed/X_test.csv", index=False)

y_train.to_csv("../data/processed/y_train.csv", index=False)
y_test.to_csv("../data/processed/y_test.csv", index=False)

print("Processed datasets saved successfully!")

Processed datasets saved successfully!


## Summary

The raw NASA Exoplanet Archive has now been transformed into a machine learning dataset through a reproducible preprocessing pipeline.

The preprocessing workflow:

- Selected scientifically meaningful stellar features.
- Removed duplicate planetary observations.
- Constructed a binary gas giant classification target.
- Excluded features with excessive missing data.
- Prevented data leakage by splitting the dataset before learning preprocessing statistics.
- Imputed missing values using medians learned exclusively from the training data.

The processed training and testing datasets are now ready for model development.